# MySQL to S3 Data Ingestion Notebook

This notebook demonstrates how to use the MySQL to S3 data ingestion script to extract data from MySQL databases and upload them as Parquet files to S3.

## Features
- ✅ MySQL database connectivity
- ✅ Multiple table support with configurable batch processing
- ✅ S3 integration for Parquet file storage
- ✅ Comprehensive logging and error handling
- ✅ Progress tracking with visual progress bars
- ✅ JSON-based configuration management

## 1. Install Required Dependencies

First, make sure all required packages are installed:

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas sqlalchemy pymysql boto3 pyarrow tqdm numpy python-dateutil

## 2. Import the Ingestion Module

In [ ]:
# Import the ingestion module
from mysql_to_s3_ingestion import MySQLToS3Ingestion, load_config, create_sample_config, display_results_summary
import json
import os
from datetime import datetime

## 3. Configuration Setup

### Option 1: Create a Sample Configuration File

In [ ]:
# Create a sample configuration file
create_sample_config("my_ingestion_config.json")

print("\n⚠️ Important: Please edit 'my_ingestion_config.json' with your actual database and S3 credentials before proceeding.")

### Option 2: Define Configuration Directly in Notebook

In [ ]:
# Define configuration directly (update with your actual credentials)
config = {
    "mysql": {
        "host": "your-mysql-host.com",
        "port": 3306,
        "database": "your_database",
        "username": "your_username",
        "password": "your_password"
    },
    "s3": {
        "bucket": "your-s3-bucket",
        "region": "us-west-2",
        "source_alias": "mysql_prod",
        "access_key_id": "your_access_key",
        "secret_access_key": "your_secret_key"
    },
    "tables": [
        {
            "name": "users",
            "batch_size": 10000
        },
        {
            "name": "orders",
            "batch_size": 5000
        }
    ],
    "logging": {
        "log_dir": "./logs"
    }
}

print("✅ Configuration defined. Update the values above with your actual credentials.")

### Option 3: Load Configuration from File

In [ ]:
# Load configuration from file (if you've created and edited the config file)
# config = load_config("my_ingestion_config.json")
# print("✅ Configuration loaded from file")

## 4. Initialize the Ingestion Engine

In [ ]:
# Initialize the ingestion engine
try:
    ingestion = MySQLToS3Ingestion(config)
    print("🚀 Ingestion engine initialized successfully!")
except Exception as e:
    print(f"❌ Failed to initialize ingestion engine: {str(e)}")
    print("\n💡 Make sure to update your configuration with valid credentials.")

## 5. Test Individual Table Processing (Optional)

You can test processing a single table before running the full ingestion:

In [ ]:
# Test processing a single table
if 'ingestion' in locals():
    test_table_config = {
        "name": "users",  # Update with an actual table name from your database
        "batch_size": 1000
    }
    
    print(f"🧪 Testing processing of table: {test_table_config['name']}")
    success = ingestion.process_table(test_table_config)
    
    if success:
        print("✅ Test table processed successfully!")
    else:
        print("❌ Test table processing failed.")
else:
    print("⚠️ Ingestion engine not initialized. Please run the previous cell first.")

## 6. Run Full Ingestion Process

In [ ]:
# Run the complete ingestion process
if 'ingestion' in locals():
    print("🚀 Starting full ingestion process...")
    print("This may take some time depending on the size of your tables.\n")
    
    # Run the ingestion
    results = ingestion.run_ingestion()
    
    # Display formatted results
    display_results_summary(results)
    
else:
    print("⚠️ Ingestion engine not initialized. Please run the initialization cell first.")

## 7. View Detailed Results

In [ ]:
# Analyze results in detail
if 'results' in locals():
    import pandas as pd
    
    # Create a results DataFrame for better visualization
    results_df = pd.DataFrame([
        {"Table": table_name, "Status": "✅ Success" if success else "❌ Failed"}
        for table_name, success in results.items()
    ])
    
    print("📊 Detailed Results:")
    display(results_df)
    
    # Summary statistics
    total_tables = len(results)
    successful_tables = sum(1 for success in results.values() if success)
    success_rate = (successful_tables / total_tables) * 100 if total_tables > 0 else 0
    
    print(f"\n📈 Success Rate: {success_rate:.1f}% ({successful_tables}/{total_tables})")
    
else:
    print("⚠️ No results available. Please run the ingestion process first.")

## 8. Verify S3 Uploads (Optional)

Check if files were successfully uploaded to S3:

In [ ]:
# List files in S3 bucket to verify uploads
if 'ingestion' in locals() and hasattr(ingestion, 's3_client'):
    try:
        bucket_name = config['s3']['bucket']
        prefix = f"landing/{config['s3'].get('source_alias', 'mysql_source')}/"
        
        response = ingestion.s3_client.list_objects_v2(
            Bucket=bucket_name,
            Prefix=prefix,
            MaxKeys=20
        )
        
        if 'Contents' in response:
            print(f"📁 Recent files in S3 bucket '{bucket_name}':")
            for obj in response['Contents']:
                size_mb = obj['Size'] / (1024 * 1024)
                print(f"   📄 {obj['Key']} ({size_mb:.2f} MB) - {obj['LastModified']}")
        else:
            print(f"📭 No files found in bucket '{bucket_name}' with prefix '{prefix}'")
            
    except Exception as e:
        print(f"❌ Error listing S3 objects: {str(e)}")
else:
    print("⚠️ S3 client not available. Please run the ingestion process first.")

## 9. Clean Up Resources

In [ ]:
# Clean up database connections
if 'ingestion' in locals():
    ingestion.close_connections()
    print("🔌 Database connections closed.")
else:
    print("ℹ️ No active connections to close.")

## 10. Additional Utilities

### Customize S3 Object Structure

The script follows the structure: `landing/source_alias/database_name/table_name/job_date/table_name.parquet`

You can customize this by modifying the `source_alias` in your configuration or by extending the `_generate_s3_key` method.

In [ ]:
# Example: Generate custom S3 key
from datetime import datetime

def generate_custom_s3_key(source_alias, database_name, table_name, job_date=None):
    """Generate a custom S3 key with your preferred structure."""
    if job_date is None:
        job_date = datetime.now().strftime('%Y-%m-%d')
    
    # Example custom structure
    return f"data-lake/{source_alias}/{database_name}/{table_name}/date={job_date}/{table_name}.parquet"

# Example usage
custom_key = generate_custom_s3_key("prod_mysql", "ecommerce", "orders", "2024-01-15")
print(f"Custom S3 Key: {custom_key}")

### Logging and Monitoring

The script creates detailed logs in the specified log directory. You can review these logs for troubleshooting:

In [ ]:
# List recent log files
import os
import glob

log_dir = config.get('logging', {}).get('log_dir', './logs')

if os.path.exists(log_dir):
    log_files = glob.glob(os.path.join(log_dir, "*.log"))
    log_files.sort(key=os.path.getmtime, reverse=True)
    
    print(f"📋 Recent log files in '{log_dir}':")
    for log_file in log_files[:5]:  # Show last 5 log files
        file_size = os.path.getsize(log_file)
        mod_time = datetime.fromtimestamp(os.path.getmtime(log_file))
        print(f"   📄 {os.path.basename(log_file)} ({file_size} bytes) - {mod_time}")
else:
    print(f"📭 Log directory '{log_dir}' not found.")

## Troubleshooting Tips

### Common Issues and Solutions:

1. **Connection Issues:**
   - Verify MySQL host, port, username, and password
   - Check firewall settings and network connectivity
   - Ensure MySQL user has appropriate permissions

2. **AWS/S3 Issues:**
   - Verify AWS credentials (access key and secret key)
   - Check S3 bucket name and region
   - Ensure IAM user has S3 write permissions

3. **Memory Issues:**
   - Reduce batch_size for large tables
   - Process tables individually if needed
   - Monitor system memory usage

4. **Performance Optimization:**
   - Adjust batch_size based on available memory
   - Use table indexes for better query performance
   - Consider running during off-peak hours

### Environment Variables (Alternative to Configuration File):

You can also use environment variables for sensitive information:

```bash
export MYSQL_HOST="your-mysql-host.com"
export MYSQL_USERNAME="your_username"
export MYSQL_PASSWORD="your_password"
export AWS_ACCESS_KEY_ID="your_access_key"
export AWS_SECRET_ACCESS_KEY="your_secret_key"
export S3_BUCKET="your-s3-bucket"
```